In [ ]:
# Navigate to the project directory and activate the virtual environment:
# cd /path/to/Research_Caspase-4

# source compound/bin/activate
# # or equivalently
# . compound/bin/activate


## 1. Installing the packages

In [ ]:
%pip install transformers torch scikit-learn pandas
## installing packages
%pip uninstall rdkit-pypi -y
%pip install rdkit-pypi
%pip install "numpy<2"
%pip install scikit-learn
%pip install tqdm pandas
%pip install matplotlib
%pip install seaborn


## 2. Importing the packages

In [ ]:
import tqdm
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit.Chem import PandasTools
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
import xml.etree.ElementTree as ET

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score


In [ ]:
# Load the data file
df = pd.read_csv('Screening energy VINA_caspase-4.csv')
print(df.head(10))


## 3. Extracting drugbank_ids

In [ ]:

# Extract leading DrugBank ID: 'DB' followed by digits
df["drugbank_id"] = df["filename"].str.extract(r'^(DB\d+)')

# Optional: check
#print(df[["filename", "drugbank_id"]].head())

# Save with IDs
df.to_csv("screening_with_drugbank_ids.csv", index=False)
print(df.head())


## 4. Extracting molecular descriptors for each DrugBank ID

In [ ]:

# List of DrugBank IDs to retrieve
drugbank_ids = df['drugbank_id'].unique().tolist()
#drugbank_ids = {'DB11742', 'DB06684', 'DB00210'}
#drugbank_ids = list(drugbank_ids)

# Load and parse the DrugBank XML file
tree = ET.parse('drugbank.xml')
root = tree.getroot()

# Namespace for DrugBank XML
ns = {'db': 'http://www.drugbank.ca'}

# Function to retrieve molecular descriptors for a given DrugBank ID
def get_molecular_descriptors(drugbank_id):
    for drug in root.findall('db:drug', ns):
        db_id = drug.find('db:drugbank-id', ns).text
        if db_id == drugbank_id:
            # Get the SMILES string from the calculated properties
            smiles_element = drug.find('db:calculated-properties/db:property[db:kind="SMILES"]/db:value', ns)
            if smiles_element is not None:
                smiles = smiles_element.text
                # Create the molecule from SMILES
                mol = Chem.MolFromSmiles(smiles)
                if mol:
                    mol_weight = Descriptors.MolWt(mol)
                    logp = Descriptors.MolLogP(mol)
                    HBA =  Descriptors.NumHAcceptors(mol)
                    HBD =  Descriptors.NumHDonors(mol)
                    RotBonds = Descriptors.NumRotatableBonds(mol)
                    AromaticRings = Descriptors.NumAromaticRings(mol)
                    return smiles, mol_weight, logp, HBA, HBD, RotBonds, AromaticRings
    return None, None, None, None, None, None, None

# Create a DataFrame to store the results
results = []

# Retrieve and store molecular descriptors for each DrugBank ID
for db_id in drugbank_ids:
    smiles, mol_weight, logp, HBA, HBD, RotBonds, AromaticRings = get_molecular_descriptors(db_id)
    if smiles:
        results.append({
            'drugbank_id': db_id,
            'smiles': smiles,
            'molecular_weight': mol_weight,
            'logp': logp,
            'HBA': HBA,
            'HBD': HBD,
            'RotBonds': RotBonds,
            'AromaticRings': AromaticRings 
        })
    else:
        print(f"SMILES not found for DrugBank ID: {db_id}")

# Convert results to a DataFrame
df = pd.DataFrame(results)

# Display the results
#print(df)
print(df.head(5))



In [ ]:
# Rows that contain at least one NaN in any column
nan_rows = df[df.isna().any(axis=1)]
print(nan_rows)
print("Number of rows with NaN:", len(nan_rows))



In [ ]:
print(df.columns)
print(df.head())


## 5. Extracting pIC50 values from chembl_ids

In [ ]:
import chembl_downloader
import pandas as pd
import numpy as np
from rdkit import Chem

# ===== 0) df must already exist with columns: drugbank_id, smiles =====

# 1) Map DrugBank SMILES -> ChEMBL chembl_id
sql_map = """
SELECT
  md.chembl_id,
  cs.canonical_smiles
FROM molecule_dictionary md
JOIN compound_structures cs
  ON md.molregno = cs.molregno;
"""
chembl_mols = chembl_downloader.query(sql_map)

def canon_smiles(smi):
    mol = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(mol, canonical=True) if mol else None

df["canonical_smiles"] = df["smiles"].apply(canon_smiles)

df = df.merge(
    chembl_mols.rename(columns={"canonical_smiles": "chembl_canonical_smiles"}),
    left_on="canonical_smiles",
    right_on="chembl_canonical_smiles",
    how="left",
)

df = df.drop(columns=["chembl_canonical_smiles"])
print("Frac with mapped chembl_id:", df["chembl_id"].notna().mean())

# 2) Pull IC50 activities for those compounds (any target)
chembl_ids = df["chembl_id"].dropna().unique().tolist()
if not chembl_ids:
    raise ValueError("No chembl_id values in df after mapping.")

chembl_ids_sql = ",".join([f"'{cid}'" for cid in chembl_ids])

sql_act = f"""
SELECT
    md.chembl_id AS compound_chembl_id,
    cs.canonical_smiles,
    act.standard_value,
    act.standard_units,
    act.standard_relation,
    act.standard_type,
    act.pchembl_value,
    act.data_validity_comment,
    a.assay_type,
    td.chembl_id AS target_chembl_id,
    td.pref_name
FROM activities act
JOIN assays a
  ON act.assay_id = a.assay_id
JOIN target_dictionary td
  ON a.tid = td.tid
JOIN molecule_dictionary md
  ON act.molregno = md.molregno
JOIN compound_structures cs
  ON md.molregno = cs.molregno
WHERE md.chembl_id IN ({chembl_ids_sql})
  AND act.standard_type = 'IC50'
  AND act.standard_units IN ('nM','uM','M')
  AND act.standard_relation = '='
  AND a.assay_type = 'B';
"""
act = chembl_downloader.query(sql_act)
print("Raw IC50 rows (all targets):", len(act))

# 3) Clean and compute pIC50
mask_valid = (
    act["standard_value"].notna() &
    ((act["data_validity_comment"].isna()) |
     (act["data_validity_comment"].isin(["Manually validated"]))
    )
)
act = act.loc[mask_valid].copy()
act["standard_value"] = act["standard_value"].astype(float)

def ic50_to_pIC50(val, unit):
    if unit == "nM":
        ic50_M = val * 1e-9
    elif unit == "uM":
        ic50_M = val * 1e-6
    elif unit == "M":
        ic50_M = val
    else:
        return np.nan
    if ic50_M <= 0:
        return np.nan
    return -np.log10(ic50_M)

act["pIC50_calc"] = act.apply(
    lambda row: ic50_to_pIC50(row["standard_value"], row["standard_units"]),
    axis=1,
)

act["pIC50"] = act["pchembl_value"].fillna(act["pIC50_calc"])
act = act.dropna(subset=["pIC50"])
act["pIC50"] = act["pIC50"].astype(float)
print("Rows with usable pIC50:", len(act))

# 4) Aggregate per compound (across all targets) and merge back
pic50_per_compound = (
    act.groupby("compound_chembl_id")
       .agg(pIC50=("pIC50", "mean"))
       .reset_index()
)

df = df.merge(
    pic50_per_compound,
    left_on="chembl_id",
    right_on="compound_chembl_id",
    how="left",
)

df = df.drop(columns=["compound_chembl_id"])
print(df[["drugbank_id", "chembl_id", "pIC50"]].head())
print("Frac with any-target pIC50:", df["pIC50"].notna().mean())


In [ ]:
        # must contain 'smiles' and pIC50/label

# 2) Filter to rows with valid SMILES (and label if needed)
df_model = df.dropna(subset=["smiles"]).reset_index(drop=True)

# 3) Create smiles_list (and y if regression/classification)
smiles_list = df_model["smiles"].tolist()
print("Number of SMILES:", len(smiles_list))


## 6. Getting the smiles embedding using ChemBERTa

In [ ]:
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "seyonec/ChemBERTa-zinc-base-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
chemberta = AutoModel.from_pretrained(model_name).to(device)
chemberta.eval()
for p in chemberta.parameters():
    p.requires_grad = False

@torch.no_grad()
def embed_smiles(smiles_batch, max_length=128):
    enc = tokenizer(
        smiles_batch,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    outputs = chemberta(input_ids=input_ids, attention_mask=attention_mask)
    token_embeddings = outputs.last_hidden_state  # (batch, seq_len, hidden)
    mask = attention_mask.unsqueeze(-1)           # (batch, seq_len, 1)
    sum_emb = (token_embeddings * mask).sum(dim=1)
    len_emb = mask.sum(dim=1).clamp(min=1)
    mean_embeddings = sum_emb / len_emb
    return mean_embeddings.cpu().numpy()         # (batch, hidden_dim)


batch_size = 32
emb_list = []

print("Number of SMILES:", len(smiles_list))  # should be > 0

for i in range(0, len(smiles_list), batch_size):
    batch = smiles_list[i:i+batch_size]
    emb = embed_smiles(batch)
    emb_list.append(emb)

if len(emb_list) == 0:
    raise ValueError("emb_list is empty – check smiles_list")

X = np.vstack(emb_list)   # shape: (N_mols, hidden_dim)
print("Embedding matrix shape:", X.shape)


## 7. Training the classifier on active vs in-active compounds

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

# 1) Classification labels at pIC50 >= 8
pIC50_thresh = 8.2
df_cls = df.dropna(subset=["smiles", "pIC50"]).copy()
df_cls["active"] = (df_cls["pIC50"] >= pIC50_thresh).astype(int)

# 2) Descriptor matrix (must exist in df)
desc_cols = ["molecular_weight", "logp", "HBA", "HBD", "RotBonds", "AromaticRings"]
X_desc = df_cls[desc_cols].values  # shape: (n_mols, 6)

# 3) SMILES embeddings in the same order
smiles_list = df_cls["smiles"].tolist()

emb_list = []
batch_size = 32
for i in range(0, len(smiles_list), batch_size):
    batch = smiles_list[i:i+batch_size]
    emb = embed_smiles(batch)      # shape: (batch_size, emb_dim)
    emb_list.append(emb)

X_emb = np.vstack(emb_list)        # shape: (n_mols, emb_dim)

# 4) Combine embeddings + descriptors
X_combined = np.hstack([X_emb, X_desc])  # shape: (n_mols, emb_dim + 6)
y_cls = df_cls["active"].values

# 5) Train / test split and RF training
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y_cls, test_size=0.2, random_state=0, stratify=y_cls
)

clf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    class_weight="balanced",
    n_jobs=-1,
    random_state=0,
)
clf.fit(X_train, y_train)

# 6) Metrics on test set
y_prob = clf.predict_proba(X_test)[:, 1]
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

# 7) Enrichment on test set (e.g., top 10%)
overall_active = y_test.mean()
order = np.argsort(-y_prob)
y_test_sorted = y_test[order]

top_frac = 0.10
k = int(len(y_test_sorted) * top_frac)
top_active = y_test_sorted[:k].mean()
enrichment = top_active / overall_active if overall_active > 0 else np.nan

print(f"Overall active fraction: {overall_active:.3f}")
print(f"Active fraction in top {top_frac*100:.0f}%: {top_active:.3f}")
print(f"Enrichment factor: {enrichment:.2f}")

# 8) If you want scores for ALL molecules in df_cls
y_score_all = clf.predict_proba(X_combined)[:, 1]
df_ranked = df_cls.copy()
df_ranked["pred_active_prob"] = y_score_all
df_ranked = df_ranked.sort_values("pred_active_prob", ascending=False)

# 9) Top 10 compounds by predicted activity probability
top10 = df_ranked.head(10).copy()

# Just to be safe, ensure pIC50 is float
top10["pIC50"] = top10["pIC50"].astype(float)

# Print as a neat table
cols_to_show = ["drugbank_id", "pIC50", "pred_active_prob",
                "molecular_weight", "logp"]
print(top10[cols_to_show].to_string(index=False))


##  8. Random Forest Regression (train the regressor on active compunds)

Accuracy: 0.949 – About 95% of test compounds are classified correctly.
​

ROC‑AUC: 0.722 – The model has moderate discrimination, ranking actives above inactives substantially better than random (0.5) but not near perfect (1.0).
​

Overall active fraction: 0.051 – Only ~5.1% of the test set are actives, so the task is highly imbalanced and early enrichment is the key metric.
​

Active fraction in top 10%: 0.176 – In the top‑scored 10% of compounds, 17.6% are actives, so hits are more concentrated near the top of the ranking.
​

Enrichment factor: 3.47 – The top 10% is about 3.5× richer in actives than random selection, which is a good EF10 for a ligand‑based model on noisy data.


For our pIC50 model, the input to the regression is the feature vector for each compound (e.g., SMILES embedding ± descriptors), and the output is a 
continuous pIC50 value predicted by the model.

In [ ]:
# top10 = df_ranked.head(10)[["drugbank_id", "smiles", "pIC50", "pred_active_prob"]]
# print(top10)

In [ ]:
# 0) Make sure X_combined has the same order as df_cls
# (this should be true where you originally built X_combined)

# 1) Add a position index into df_cls that matches X_combined rows
df_cls = df_cls.reset_index(drop=True)
df_cls["row_id"] = np.arange(len(df_cls))

# 2) Build a small helper DataFrame linking drugbank_id to row_id
idx_map = df_cls[["drugbank_id", "row_id", "pIC50"]].copy()

# 3) Select actives for regression using pIC50 threshold
pIC50_thr = 8.2
df_regtrain = idx_map[idx_map["pIC50"] >= pIC50_thr].copy()

# Now get X_reg via row_id, which is guaranteed in range
X_reg = X_combined[df_regtrain["row_id"].values]
y_reg = df_regtrain["pIC50"].astype(float).values

# 4) Train/test split and fit regressor
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=0
)

rf_reg = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    n_jobs=-1,
    random_state=0,
)
rf_reg.fit(X_train_r, y_train_r)

# 5) Predict pIC50 for top10, aligned via row_id too
top10 = df_ranked.sort_values("pred_active_prob", ascending=False).head(10).copy()
top10 = top10.merge(idx_map[["drugbank_id", "row_id"]], on="drugbank_id", how="left")

X_top10 = X_combined[top10["row_id"].values]
top10["pIC50_pred"] = rf_reg.predict(X_top10)

print(top10[["drugbank_id", "pIC50", "pIC50_pred", "pred_active_prob"]])



The classifier takes each molecule’s features (from X_combined) and learns to predict how likely that molecule is to be “active” (e.g., pIC50 above some threshold).

During training, its inputs are the feature vectors for all molecules and the target labels (active vs. inactive) derived from your pIC50 threshold.

After training, for any new or existing molecule, the classifier outputs a probability of being active (pred_active_prob) and you use this to rank molecules.

You then pick the top‑ranked 10 molecules (highest pred_active_prob) as the most promising candidates.

The regressor’s input is the feature vectors (X_reg) only for molecules that have real potency values and are above the chosen pIC50 threshold (e.g., ≥ 8.2), and its target output during training is their continuous pIC50 values (y_reg).

After training, you feed the regressor the feature vectors of the top‑10 classifier hits (X_top10), and its output is a predicted pIC50 (pIC50_pred) for each of those top‑10 molecules.

Important: Many studies restrict the regression set to compounds above a certain potency threshold (e.g., only “actives”) so the model focuses on learning structure–potency relationships within a chemically relevant, high‑quality activity range.

Using only molecules with real, reliable pIC50 values avoids training on noisy or missing labels and generally improves the statistical quality and interpretability of the QSAR model.

## 9. Plotting the error for actual versus predicted pIC50 values for top 10 compounds

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

# Assume top10 has columns: pIC50 (actual) and pIC50_pred (predicted)
errors = np.abs(top10["pIC50_pred"] - top10["pIC50"]).values
labels = top10["drugbank_id"].values

# Optional: RMSE over these 10 points
rmse = np.sqrt(mean_squared_error(top10["pIC50"], top10["pIC50_pred"]))

x = np.arange(len(errors))

# plt.figure(figsize=(6, 4))
# plt.plot(x, errors, marker="o")
# plt.axhline(rmse, color="red", linestyle="--", label=f"RMSE (top 10) = {rmse:.3f}")

# plt.xticks(x, labels, rotation=45, ha="right")
# plt.ylabel("|predicted − actual| pIC50")
# plt.title("Absolute error between actual and predicted pIC50 (top 10)")
# plt.legend()
# plt.tight_layout()
# plt.show()

plt.figure(figsize=(6, 4))
plt.plot(x, errors, marker="o")
plt.axhline(rmse, color="red", linestyle="--", label=f"RMSE (top 10) = {rmse:.3f}")

plt.xticks(x, labels, rotation=45, ha="right")
plt.ylabel("|predicted − actual| pIC50")
plt.ylim(0, 1)          # force y-axis from 0 to 2
plt.title("Absolute error between actual and predicted pIC50 (top 10)")
plt.legend()
plt.tight_layout()
plt.show()



## SA Score

In [ ]:
from rdkit import Chem
import sascorer  # this now imports the local file

def compute_sa(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return sascorer.calculateScore(mol)

top10["SA_score"] = top10["smiles"].apply(compute_sa)

print(top10[["drugbank_id", "pIC50", "SA_score"]].to_string(index=False))




SA_score around 1–4 → relatively easy to synthesize.

SA_score around 4–6 → moderate difficulty.

SA_score above 6 → increasingly hard or unlikely to be practical to make in the lab.

## Table of molecular properties of screened compunds

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# 0) Rebuild top10 directly from df_ranked, pulling needed columns
top10 = df_ranked.head(10)[[
    "drugbank_id",
    "smiles",
    "molecular_weight",
    "logp",
    "HBA",
    "HBD"
]].copy()

# 1) Build RDKit mol objects
top10["mol"] = top10["smiles"].apply(
    lambda s: Chem.MolFromSmiles(s) if pd.notna(s) else None
)

def mol_formula(m):
    return rdMolDescriptors.CalcMolFormula(m) if m is not None else None

def mol_volume(m):
    return Descriptors.MolMR(m) if m is not None else None  # volume proxy

def num_atoms(m):
    return m.GetNumAtoms() if m is not None else None

def num_stereo_centers(m):
    if m is None:
        return None
    return len(Chem.FindMolChiralCenters(m, includeUnassigned=True))

# New: molar refractivity, polarizability, TPSA
def mol_refractivity(m):
    return Descriptors.MolMR(m) if m is not None else None

def mol_polarizability(m, k=1.0):
    mr = mol_refractivity(m)
    return k * mr if mr is not None else None

def mol_tpsa(m):
    return rdMolDescriptors.CalcTPSA(m) if m is not None else None

top10["MolFormula"]        = top10["mol"].apply(mol_formula)
top10["MolVolume"]         = top10["mol"].apply(mol_volume)
top10["NumAtoms"]          = top10["mol"].apply(num_atoms)
top10["StereoCenters"]     = top10["mol"].apply(num_stereo_centers)
top10["MolRefractivity"]   = top10["mol"].apply(mol_refractivity)
top10["MolPolarizability"] = top10["mol"].apply(mol_polarizability)
top10["TPSA"]              = top10["mol"].apply(mol_tpsa)

# 2) Build property-by-compound table (with molar properties + TPSA)
prop_table = pd.DataFrame({
    "Properties": [
        "Mol. Formula",
        "Mol. Weight (g/mol)",
        "Molar Polarizability (arb.)",
        "Molar Refractivity (cm^3)",
        "Log P",
        "HBD",
        "HBA",
        "Polar Surface Area (Å^2)",
        "Molecular Volume (A^3)",
        "No. stereo centers",
        "No. of atoms",
        "Lipinski Rule"
    ]
})

col_ids = [f"C{i+1}" for i in range(len(top10))]

for idx, row in top10.reset_index(drop=True).iterrows():
    col_name = col_ids[idx]

    lipinski_ok = (
        (row["molecular_weight"] <= 500) and
        (row["HBD"] <= 5) and
        (row["HBA"] <= 10) and
        (row["logp"] <= 5)
    )
    lipinski_str = "Yes" if lipinski_ok else "No"

    prop_table[col_name] = [
        row["MolFormula"],
        f"{row['molecular_weight']:.2f}",
        f"{row['MolPolarizability']:.2f}" if row["MolPolarizability"] is not None else "",
        f"{row['MolRefractivity']:.2f}"   if row["MolRefractivity"]   is not None else "",
        f"{row['logp']:.2f}",
        int(row["HBD"]),
        int(row["HBA"]),
        f"{row['TPSA']:.2f}"              if row["TPSA"] is not None else "",
        f"{row['MolVolume']:.2f}"         if row["MolVolume"] is not None else "",
        row["StereoCenters"] if row["StereoCenters"] is not None else "",
        row["NumAtoms"] if row["NumAtoms"] is not None else "",
        lipinski_str,
    ]

# 3) Output to copy into Word
print(prop_table.to_string(index=False))
prop_table.to_csv("top10_molecular_properties.csv", index=False)


## Table with more properties

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# 0) Rebuild top10 directly from df_ranked, pulling needed columns
top10 = df_ranked.head(10)[[
    "drugbank_id",
    "smiles",
    "molecular_weight",
    "logp",
    "HBA",
    "HBD"
]].copy()

# 1) Build RDKit mol objects
top10["mol"] = top10["smiles"].apply(
    lambda s: Chem.MolFromSmiles(s) if pd.notna(s) else None
)

def mol_formula(m):
    return rdMolDescriptors.CalcMolFormula(m) if m is not None else None

def mol_volume(m):
    return Descriptors.MolMR(m) if m is not None else None  # proxy

def num_atoms(m):
    return m.GetNumAtoms() if m is not None else None

def num_stereo_centers(m):
    if m is None:
        return None
    return len(Chem.FindMolChiralCenters(m, includeUnassigned=True))

# New: molar refractivity, polarizability, TPSA
def mol_refractivity(m):
    return Descriptors.MolMR(m) if m is not None else None

def mol_polarizability(m, k=1.0):
    mr = mol_refractivity(m)
    return k * mr if mr is not None else None

def mol_tpsa(m):
    return rdMolDescriptors.CalcTPSA(m) if m is not None else None

top10["MolFormula"]        = top10["mol"].apply(mol_formula)
top10["MolVolume"]         = top10["mol"].apply(mol_volume)
top10["NumAtoms"]          = top10["mol"].apply(num_atoms)
top10["StereoCenters"]     = top10["mol"].apply(num_stereo_centers)
top10["MolRefractivity"]   = top10["mol"].apply(mol_refractivity)
top10["MolPolarizability"] = top10["mol"].apply(mol_polarizability)
top10["TPSA"]              = top10["mol"].apply(mol_tpsa)

# ---- heuristic PK tendencies from logP and TPSA ----
def lipophilicity_class(logp):
    if logp < 1:   return "very low"
    if logp < 3:   return "moderate"
    if logp < 5:   return "high"
    return "very high"      # relates to Ro5 logP≤5.[web:63][web:68]

def polarity_class(psa):
    if psa < 60:    return "low"
    if psa < 90:    return "moderate"
    if psa < 140:   return "high"
    return "very high"      # PSA >140 Å² often limits permeability.[web:64]

def vd_tendency(logp, psa):
    if logp >= 3.5 and psa < 75:
        return "likely high"
    if logp < 1 and psa > 90:
        return "low"
    return "moderate"       # rule‑of‑thumb using logP/TPSA vs distribution.[web:52][web:66]

def clearance_tendency(logp, psa):
    if logp < 2 and psa > 90:
        return "high"
    if logp > 4.5 and psa < 75:
        return "low"
    return "moderate"       # more polar → faster clearance on average.[web:52][web:69]

def half_life_tendency(vd_class, cl_class):
    if vd_class == "likely high" and cl_class == "low":
        return "long"
    if vd_class == "low" and cl_class == "high":
        return "short"
    return "intermediate"   # links t½ qualitatively to Vd and CL.[web:50]

top10["Lipophilicity"]      = top10["logp"].apply(lipophilicity_class)
top10["Polarity"]           = top10["TPSA"].apply(polarity_class)
top10["Vd_tendency"]        = top10.apply(lambda r: vd_tendency(r["logp"], r["TPSA"]), axis=1)
top10["Clearance_tendency"] = top10.apply(lambda r: clearance_tendency(r["logp"], r["TPSA"]), axis=1)
top10["Half_life_tendency"] = top10.apply(
    lambda r: half_life_tendency(r["Vd_tendency"], r["Clearance_tendency"]), axis=1
)

# 2) Build property-by-compound table (add new PK‑like rows)
prop_table = pd.DataFrame({
    "Properties": [
        "Mol. Formula",
        "Mol. Weight (g/mol)",
        "Molar Polarizability (arb.)",
        "Molar Refractivity (cm^3)",
        "Log P",
        "HBD",
        "HBA",
        "Polar Surface Area (Å^2)",
        "Molecular Volume (A^3)",
        "No. stereo centers",
        "No. of atoms",
        "Lipinski Rule",
        "Lipophilicity (qual.)",
        "Polarity (qual.)",
        "Distribution Vd (qual.)",
        "Clearance (qual.)",
        "Half-life (qual.)",
    ]
})

col_ids = [f"C{i+1}" for i in range(len(top10))]

for idx, row in top10.reset_index(drop=True).iterrows():
    col_name = col_ids[idx]

    lipinski_ok = (
        (row["molecular_weight"] <= 500) and
        (row["HBD"] <= 5) and
        (row["HBA"] <= 10) and
        (row["logp"] <= 5)
    )
    lipinski_str = "Yes" if lipinski_ok else "No"

    prop_table[col_name] = [
        row["MolFormula"],
        f"{row['molecular_weight']:.2f}",
        f"{row['MolPolarizability']:.2f}" if row["MolPolarizability"] is not None else "",
        f"{row['MolRefractivity']:.2f}"   if row["MolRefractivity"]   is not None else "",
        f"{row['logp']:.2f}",
        int(row["HBD"]),
        int(row["HBA"]),
        f"{row['TPSA']:.2f}"              if row["TPSA"] is not None else "",
        f"{row['MolVolume']:.2f}"         if row["MolVolume"] is not None else "",
        row["StereoCenters"] if row["StereoCenters"] is not None else "",
        row["NumAtoms"] if row["NumAtoms"] is not None else "",
        lipinski_str,
        row["Lipophilicity"],
        row["Polarity"],
        row["Vd_tendency"],
        row["Clearance_tendency"],
        row["Half_life_tendency"],
    ]

# 3) Output to copy into Word
print(prop_table.to_string(index=False))
prop_table.to_csv("top10_molecular_properties_with_PK_like_cols.csv", index=False)


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# 0) Rebuild top10 directly from df_ranked, pulling needed columns
top10 = df_ranked.head(10)[[
    "drugbank_id",
    "smiles",
    "molecular_weight",
    "logp",
    "HBA",
    "HBD"
]].copy()

# 1) Build RDKit mol objects
top10["mol"] = top10["smiles"].apply(
    lambda s: Chem.MolFromSmiles(s) if pd.notna(s) else None
)

def mol_formula(m):
    return rdMolDescriptors.CalcMolFormula(m) if m is not None else None

def mol_volume(m):
    return Descriptors.MolMR(m) if m is not None else None  # proxy

def num_atoms(m):
    return m.GetNumAtoms() if m is not None else None

def num_stereo_centers(m):
    if m is None:
        return None
    return len(Chem.FindMolChiralCenters(m, includeUnassigned=True))

# New: molar refractivity, polarizability, TPSA
def mol_refractivity(m):
    return Descriptors.MolMR(m) if m is not None else None

def mol_polarizability(m, k=1.0):
    mr = mol_refractivity(m)
    return k * mr if mr is not None else None

def mol_tpsa(m):
    return rdMolDescriptors.CalcTPSA(m) if m is not None else None

top10["MolFormula"]        = top10["mol"].apply(mol_formula)
top10["MolVolume"]         = top10["mol"].apply(mol_volume)
top10["NumAtoms"]          = top10["mol"].apply(num_atoms)
top10["StereoCenters"]     = top10["mol"].apply(num_stereo_centers)
top10["MolRefractivity"]   = top10["mol"].apply(mol_refractivity)
top10["MolPolarizability"] = top10["mol"].apply(mol_polarizability)
top10["TPSA"]              = top10["mol"].apply(mol_tpsa)

# ---- heuristic PK tendencies from logP and TPSA ----
def lipophilicity_class(logp):
    if logp < 1:   return "very low"
    if logp < 3:   return "moderate"
    if logp < 5:   return "high"
    return "very high"      # relates to Ro5 logP≤5.[web:63][web:68]

def polarity_class(psa):
    if psa < 60:    return "low"
    if psa < 90:    return "moderate"
    if psa < 140:   return "high"
    return "very high"      # PSA >140 Å² often limits permeability.[web:64]

def vd_tendency(logp, psa):
    if logp >= 3.5 and psa < 75:
        return "likely high"
    if logp < 1 and psa > 90:
        return "low"
    return "moderate"       # rule‑of‑thumb using logP/TPSA vs distribution.[web:52][web:66]

def clearance_tendency(logp, psa):
    if logp < 2 and psa > 90:
        return "high"
    if logp > 4.5 and psa < 75:
        return "low"
    return "moderate"       # more polar → faster clearance on average.[web:52][web:69]

def half_life_tendency(vd_class, cl_class):
    if vd_class == "likely high" and cl_class == "low":
        return "long"
    if vd_class == "low" and cl_class == "high":
        return "short"
    return "intermediate"   # links t½ qualitatively to Vd and CL.[web:50]

top10["Lipophilicity"]      = top10["logp"].apply(lipophilicity_class)
top10["Polarity"]           = top10["TPSA"].apply(polarity_class)
top10["Vd_tendency"]        = top10.apply(lambda r: vd_tendency(r["logp"], r["TPSA"]), axis=1)
top10["Clearance_tendency"] = top10.apply(lambda r: clearance_tendency(r["logp"], r["TPSA"]), axis=1)
top10["Half_life_tendency"] = top10.apply(
    lambda r: half_life_tendency(r["Vd_tendency"], r["Clearance_tendency"]), axis=1
)

# 2) Build property-by-compound table (add new PK‑like rows)
prop_table = pd.DataFrame({
    "Properties": [
        "Mol. Formula",
        "Mol. Weight (g/mol)",
        "Molar Polarizability (arb.)",
        "Molar Refractivity (cm^3)",
        "Log P",
        "HBD",
        "HBA",
        "Polar Surface Area (Å^2)",
        "Molecular Volume (A^3)",
        "No. stereo centers",
        "No. of atoms",
        "Lipinski Rule",
        "Lipophilicity (qual.)",
        "Polarity (qual.)",
        "Distribution Vd (qual.)",
        "Clearance (qual.)",
        "Half-life (qual.)",
    ]
})

col_ids = [f"C{i+1}" for i in range(len(top10))]

for idx, row in top10.reset_index(drop=True).iterrows():
    col_name = col_ids[idx]

    lipinski_ok = (
        (row["molecular_weight"] <= 500) and
        (row["HBD"] <= 5) and
        (row["HBA"] <= 10) and
        (row["logp"] <= 5)
    )
    lipinski_str = "Yes" if lipinski_ok else "No"

    prop_table[col_name] = [
        row["MolFormula"],
        f"{row['molecular_weight']:.2f}",
        f"{row['MolPolarizability']:.2f}" if row["MolPolarizability"] is not None else "",
        f"{row['MolRefractivity']:.2f}"   if row["MolRefractivity"]   is not None else "",
        f"{row['logp']:.2f}",
        int(row["HBD"]),
        int(row["HBA"]),
        f"{row['TPSA']:.2f}"              if row["TPSA"] is not None else "",
        f"{row['MolVolume']:.2f}"         if row["MolVolume"] is not None else "",
        row["StereoCenters"] if row["StereoCenters"] is not None else "",
        row["NumAtoms"] if row["NumAtoms"] is not None else "",
        lipinski_str,
        row["Lipophilicity"],
        row["Polarity"],
        row["Vd_tendency"],
        row["Clearance_tendency"],
        row["Half_life_tendency"],
    ]

# 3) Output to copy into Word
print(prop_table.to_string(index=False))
prop_table.to_csv("top10_molecular_properties_with_PK_like_cols.csv", index=False)


## Radar plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# pick one compound, e.g. C1
row = top10.iloc[0]   # after you added Lipophilicity, Polarity, Vd_tendency, etc.

def score_lipophilicity(label):
    mapping = {"very low": 0, "moderate": 25, "high": 50, "very high": 75}
    return mapping.get(label, 0)

def score_polarity(label):       # higher score = more polar
    mapping = {"low": 0, "moderate": 25, "high": 50, "very high": 75}
    return mapping.get(label, 0)

def score_vd(label):
    mapping = {"low": 0, "moderate": 25, "likely high": 50}
    return mapping.get(label, 0)

def score_clearance(label):
    mapping = {
        "low clearance": 0,
        "moderate clearance": 25,
        "likely high clearance": 50
    }
    return mapping.get(label, 0)

def score_half_life(label):
    mapping = {"short": 0, "intermediate": 25, "long": 50}
    return mapping.get(label, 0)

lip_score  = score_lipophilicity(row["Lipophilicity"])
tpsa_score = score_polarity(row["Polarity"])
vd_score   = score_vd(row["Vd_tendency"])
cl_score   = score_clearance(row["Clearance_tendency"])
t12_score  = score_half_life(row["Half_life_tendency"])

labels = ["LogP", "TPSA", "Vd", "CL", "t½"]
values = [lip_score, tpsa_score, vd_score, cl_score, t12_score]

# close the polygon
values = values + values[:1]
angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False)
angles = np.concatenate([angles, [angles[0]]])

fig, ax = plt.subplots(subplot_kw={"polar": True})
ax.plot(angles, values, linewidth=2)
ax.fill(angles, values, alpha=0.25)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)

# show 10, 20, 30, ... ticks
ax.set_yticks([10, 20, 30, 40, 50, 60, 70, 80])
ax.set_yticklabels(["10", "20", "30", "40", "50", "60", "70", "80"])

ax.set_title("Pharmacokinetic Property Profile")
plt.show()


In [ ]:
# print(top10.columns.tolist())
# print(top10.head(1).T)  # optional: see one row transposed


## radar plot for 10 compunds

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- 1) Create PK label columns in top10 from existing descriptors ----------

top10 = top10.copy()

def label_lipophilicity(logp):
    if logp < 1.0:
        return "very low"
    elif logp < 3.0:
        return "moderate"
    elif logp < 5.0:
        return "high"
    else:
        return "very high"

def label_polarity(mw):
    if mw < 300:
        return "high"
    elif mw < 450:
        return "moderate"
    else:
        return "low"

def label_vd(mw):
    if mw < 300:
        return "likely high"
    elif mw < 450:
        return "moderate"
    else:
        return "low"

def label_clearance(logp):
    if logp < 1.0:
        return "likely high clearance"
    elif logp < 3.0:
        return "moderate clearance"
    else:
        return "low clearance"

def label_half_life(logp):
    if logp < 1.0:
        return "short"
    elif logp < 3.0:
        return "intermediate"
    else:
        return "long"

top10["Lipophilicity"]      = top10["logp"].apply(label_lipophilicity)
top10["Polarity"]           = top10["molecular_weight"].apply(label_polarity)
top10["Vd_tendency"]        = top10["molecular_weight"].apply(label_vd)
top10["Clearance_tendency"] = top10["logp"].apply(label_clearance)
top10["Half_life_tendency"] = top10["logp"].apply(label_half_life)

# ---------- 2) Scoring functions (your original scales) ----------

def score_lipophilicity(label):
    mapping = {"very low": 0, "moderate": 25, "high": 50, "very high": 75}
    return mapping.get(label, 0)

def score_polarity(label):
    mapping = {"low": 0, "moderate": 25, "high": 50, "very high": 75}
    return mapping.get(label, 0)

def score_vd(label):
    mapping = {"low": 0, "moderate": 25, "likely high": 50}
    return mapping.get(label, 0)

def score_clearance(label):
    mapping = {
        "low clearance": 0,
        "moderate clearance": 25,
        "likely high clearance": 50,
    }
    return mapping.get(label, 0)

def score_half_life(label):
    mapping = {"short": 0, "intermediate": 25, "long": 50}
    return mapping.get(label, 0)

# ---------- 3) Radar plots for all top 10 (2 rows × 5 cols) ----------

labels = ["LogP", "TPSA", "Vd", "CL", "t½"]
angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False)
angles = np.concatenate([angles, [angles[0]]])

fig, axes = plt.subplots(2, 5, subplot_kw={"polar": True}, figsize=(14, 6))
axes = axes.ravel()

for i, (_, row) in enumerate(top10.iloc[:10].iterrows()):
    lip_score  = score_lipophilicity(row["Lipophilicity"])
    tpsa_score = score_polarity(row["Polarity"])
    vd_score   = score_vd(row["Vd_tendency"])
    cl_score   = score_clearance(row["Clearance_tendency"])
    t12_score  = score_half_life(row["Half_life_tendency"])

    values = [lip_score, tpsa_score, vd_score, cl_score, t12_score]
    values = values + values[:1]

    ax = axes[i]
    ax.plot(angles, values, linewidth=2)
    ax.fill(angles, values, alpha=0.25)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=7)

    ax.set_yticks([10, 20, 30, 40, 50, 60, 70, 80])
    ax.set_yticklabels(["10", "20", "30", "40", "50", "60", "70", "80"], fontsize=6)

    ax.set_title(str(row["drugbank_id"]), fontsize=9)

# hide any unused subplots if <10 rows
for j in range(len(top10), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Pharmacokinetic Property Profiles (Top 10 Compounds)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(5, 2, subplot_kw={"polar": True}, figsize=(10, 14))
axes = axes.ravel()  # now 10 axes in 5×2 layout

for i, (_, row) in enumerate(top10.iloc[:10].iterrows()):
    lip_score  = score_lipophilicity(row["Lipophilicity"])
    tpsa_score = score_polarity(row["Polarity"])
    vd_score   = score_vd(row["Vd_tendency"])
    cl_score   = score_clearance(row["Clearance_tendency"])
    t12_score  = score_half_life(row["Half_life_tendency"])

    values = [lip_score, tpsa_score, vd_score, cl_score, t12_score]
    values = values + values[:1]

    ax = axes[i]
    ax.plot(angles, values, linewidth=2)
    ax.fill(angles, values, alpha=0.25)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=7)

    ax.set_yticks([10, 20, 30, 40, 50, 60, 70, 80])
    ax.set_yticklabels(["10", "20", "30", "40", "50", "60", "70", "80"], fontsize=6)

    ax.set_title(str(row["drugbank_id"]), fontsize=9)

# hide any unused subplots if <10 rows
for j in range(len(top10), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Pharmacokinetic Property Profiles (Top 10 Compounds)", fontsize=14)
plt.tight_layout()
plt.show()


## dose response curve

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Take top 10 ranked by whatever score you use
top10 = df_ranked.head(10).copy()

Emax = 1.0
E0   = 0.0
h    = 1.0
conc = np.logspace(-2, 2, 200)   # 0.01–100 µM

fig, axes = plt.subplots(2, 5, figsize=(14, 6), sharex=True, sharey=True)
axes = axes.ravel()

for i, (_, row) in enumerate(top10.iterrows()):
    pIC50 = row["pIC50"]
    IC50_M  = 10**(-pIC50)
    IC50_uM = IC50_M * 1e6

    effect = E0 + (Emax - E0) * conc**h / (conc**h + IC50_uM**h)

    ax = axes[i]
    ax.semilogx(conc, effect)
    ax.set_title(str(row["drugbank_id"]), fontsize=9)
    ax.grid(True, which="both", ls="--", alpha=0.4)

    if i >= 5:
        ax.set_xlabel("Conc (µM)", fontsize=8)
    if i % 5 == 0:
        ax.set_ylabel("Effect / Emax", fontsize=8)

# hide any unused axes if <10 rows
for j in range(len(top10), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Predicted Dose–Response Curves (Top 10 Compounds)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(8, 12), sharex=True, sharey=True)
axes = axes.ravel()

colors = plt.cm.tab10(np.linspace(0, 1, len(top10)))  # 10 distinct colors

for i, (_, row) in enumerate(top10.iterrows()):
    pIC50   = row["pIC50"]
    IC50_M  = 10**(-pIC50)
    IC50_uM = IC50_M * 1e6

    effect = E0 + (Emax - E0) * conc**h / (conc**h + IC50_uM**h)

    ax = axes[i]
    ax.semilogx(conc, effect, color=colors[i], linewidth=2.5)
    ax.set_title(str(row["drugbank_id"]), fontsize=9)
    ax.grid(True, which="both", ls="--", alpha=0.4)

    # last row -> x‑label
    if i // 2 == 4:
        ax.set_xlabel("Conc (µM)", fontsize=8)
    # first column -> y‑label
    if i % 2 == 0:
        ax.set_ylabel("Effect / Emax", fontsize=8)

# hide unused axes if <10
for j in range(len(top10), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Predicted Dose–Response Curves (Top 10 Compounds)", fontsize=14)
plt.tight_layout()
plt.show()


## Integrated PK-PD simulations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Take top 10 ranked compounds
top10 = df_ranked.head(10).copy()

# Fixed PK assumptions (you can later vary per compound)
C0 = 10.0          # initial plasma concentration (µM)
t_half = 6.0       # half-life in hours
k_el = np.log(2) / t_half

Emax = 1.0
E0   = 0.0
h    = 1.0

t = np.linspace(0, 24, 200)  # time grid (0–24 h)

fig, axes = plt.subplots(2, 5, figsize=(14, 6), sharex=True)
axes = axes.ravel()

for i, (_, row) in enumerate(top10.iterrows()):
    pIC50 = row["pIC50"]
    IC50_M  = 10**(-pIC50)
    IC50_uM = IC50_M * 1e6

    # PK: concentration vs time
    C = C0 * np.exp(-k_el * t)

    # PD: effect vs time
    E = E0 + (Emax - E0) * C**h / (C**h + IC50_uM**h)

    ax = axes[i]
    ax.plot(t, C, label="Conc (µM)")
    ax.plot(t, E * C0, label="Effect (scaled)")
    ax.set_title(str(row["drugbank_id"]), fontsize=9)
    ax.grid(True, alpha=0.4)

    if i >= 5:
        ax.set_xlabel("Time (h)", fontsize=8)
    if i % 5 == 0:
        ax.set_ylabel("Value", fontsize=8)

# Hide unused axes if fewer than 10 rows
for j in range(len(top10), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("PK–PD Simulations (Top 10 Compounds)", fontsize=14)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")
plt.tight_layout()
plt.show()


They are identical because, in the code, nothing in the PK–PD calculation actually depends on which compound you are looping over.

C is computed only from C0, t_half, and t, which are the same for every compound, so the concentration–time curve is identical in all panels.

With your current parameters (Emax = 1, E0 = 0, fixed C0, and very small IC50_uM for potent compounds), the Emax term is essentially 1 for all time points and all top‑10 pIC50 values, so the effect curve is also almost flat and the same across compounds.

vary PK parameters (clearance, half‑life) or PD parameters (Emax, h, IC50) per compound to generate distinct curves.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Top 10 compounds
top10 = df_ranked.head(10).copy()

# Assign numeric half-lives between 2 and 12 h across the top 10
t_half_values = np.linspace(2, 12, len(top10))   # hours

Emax = 1.0
E0   = 0.0
h    = 1.0

C0 = 0.3                 # lower initial concentration (µM)
t  = np.linspace(0, 24, 200)

fig, axes = plt.subplots(2, 5, figsize=(14, 6), sharex=True)
axes = axes.ravel()

for i, ((_, row), t_half) in enumerate(zip(top10.iterrows(), t_half_values)):
    pIC50 = row["pIC50"]
    IC50_M  = 10**(-pIC50)
    IC50_uM = IC50_M * 1e6

    # kel from half-life
    k_el = np.log(2) / t_half

    # PK: concentration vs time
    C = C0 * np.exp(-k_el * t)

    # PD: Emax model
    E = E0 + (Emax - E0) * C**h / (C**h + IC50_uM**h)

    ax = axes[i]
    ax.plot(t, C, label="Conc (µM)")
    ax.plot(t, E * C0, label="Effect (scaled)")
    ax.set_title(f'{row["drugbank_id"]}\n t½={t_half:.1f} h', fontsize=8)
    ax.grid(True, alpha=0.4)

    if i >= 5:
        ax.set_xlabel("Time (h)", fontsize=8)
    if i % 5 == 0:
        ax.set_ylabel("Value", fontsize=8)

# Hide any unused axes
for j in range(len(top10), len(axes)):
    fig.delaxes(axes[j])

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")
fig.suptitle("PK–PD with Varying Half-lives (Top 10)", fontsize=14)
plt.tight_layout()
plt.show()


Your updated code is doing the right thing; it should now give different curves per compound because you vary the half‑life.

In simple terms, this script:

Takes the top‑10 ranked compounds.

Assigns each one a different half‑life between 2 and 12 hours, and uses that to compute a different elimination rate kel
and concentration–time curve.

Computes effect over time with an Emax model, so both concentration and effect now decay at different speeds for different compounds.

So “doing this” is exactly how you introduce PK variability and avoid identical plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# PK parameters
C0 = 10.0      # µM
t_half = 6.0   # h
k_el = np.log(2) / t_half

# PD parameters (Emax)
E0   = 0.0
Emax = 100.0   # % effect
EC50 = 1.0     # µM
h    = 1.0

t = np.linspace(0, 24, 200)

# PK: concentration vs time
C = C0 * np.exp(-k_el * t)

# PD: effect vs time (via concentration)
E = E0 + (Emax * C**h) / (EC50**h + C**h)

fig, ax1 = plt.subplots(figsize=(6, 4))

# Left y-axis: concentration
ax1.plot(t, C, color="tab:blue", label="Conc (µM)")
ax1.set_xlabel("Time (h)")
ax1.set_ylabel("Conc (µM)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

# Right y-axis: effect
ax2 = ax1.twinx()
ax2.plot(t, E, color="tab:red", label="Effect (%)")
ax2.set_ylabel("Effect (%)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

fig.suptitle("PK–PD Time Course (Single Compound)")
fig.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

top10 = df_ranked.head(10).copy()

# PK parameters (could vary per compound if you want)
C0 = 10.0      # µM
t_half = 6.0   # h
k_el = np.log(2) / t_half

# PD parameters
E0   = 0.0
Emax = 100.0   # % effect
EC50 = 1.0     # µM
h    = 1.0

t = np.linspace(0, 24, 200)

fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharex=True)
axes = axes.ravel()

for i, (_, row) in enumerate(top10.iterrows()):
    ax1 = axes[i]                     # primary axis
    ax2 = ax1.twinx()                 # secondary y-axis

    # You can still use pIC50 if you want EC50 per compound
    # pIC50 = row["pIC50"]
    # IC50_M  = 10**(-pIC50)
    # EC50    = IC50_M * 1e6

    # PK
    C = C0 * np.exp(-k_el * t)

    # PD
    E = E0 + (Emax * C**h) / (EC50**h + C**h)

    # Plot on left y-axis: concentration
    ln1 = ax1.plot(t, C, color="tab:blue", label="Conc (µM)")
    ax1.set_ylabel("Conc (µM)", fontsize=8, color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    # Plot on right y-axis: effect
    ln2 = ax2.plot(t, E, color="tab:red", label="Effect (%)")
    ax2.set_ylabel("Effect (%)", fontsize=8, color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")

    ax1.set_title(str(row["drugbank_id"]), fontsize=9)
    ax1.grid(True, alpha=0.4)

    if i >= 5:
        ax1.set_xlabel("Time (h)", fontsize=8)

# Delete any unused axes if fewer than 10 compounds
for j in range(len(top10), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("PK–PD Time Courses (Top 10 Compounds)", fontsize=14)

# Build a single legend from first subplot
lines, labels = [], []
for ax in [axes[0], axes[0].twinx()]:
    lns, lbs = ax.get_legend_handles_labels()
    lines.extend(lns)
    labels.extend(lbs)
fig.legend(lines, labels, loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

top10 = df_ranked.head(10).copy()

# Assign different half-lives (e.g. 2–12 h) across the 10 compounds
t_half_values = np.linspace(2, 12, len(top10))   # hours

# PK initial concentration
C0 = 10.0    # µM

# PD parameters
E0   = 0.0
Emax = 100.0   # % effect
EC50 = 1.0     # µM
h    = 1.0

t = np.linspace(0, 24, 200)

fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharex=True)
axes = axes.ravel()

for i, ((_, row), t_half) in enumerate(zip(top10.iterrows(), t_half_values)):
    ax1 = axes[i]
    ax2 = ax1.twinx()

    # PK: k_el from half-life
    k_el = np.log(2) / t_half
    C = C0 * np.exp(-k_el * t)

    # PD: Emax
    E = E0 + (Emax * C**h) / (EC50**h + C**h)

    ax1.plot(t, C, color="tab:blue", label="Conc (µM)")
    ax1.set_ylabel("Conc (µM)", fontsize=8, color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    ax2.plot(t, E, color="tab:red", label="Effect (%)")
    ax2.set_ylabel("Effect (%)", fontsize=8, color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")

    ax1.set_title(f'{row["drugbank_id"]}\n t½ = {t_half:.1f} h', fontsize=9)
    ax1.grid(True, alpha=0.4)

    if i >= 5:
        ax1.set_xlabel("Time (h)", fontsize=8)

# Delete unused axes if <10
for j in range(len(top10), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("PK–PD with Varying Half-lives (Top 10 Compounds)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(10, 14), sharex=True)
axes = axes.ravel()  # now len(axes) == 10

for i, ((_, row), t_half) in enumerate(zip(top10.iterrows(), t_half_values)):
    ax1 = axes[i]
    ax2 = ax1.twinx()

    k_el = np.log(2) / t_half
    C = C0 * np.exp(-k_el * t)
    E = E0 + (Emax * C**h) / (EC50**h + C**h)

    ax1.plot(t, C, color="tab:blue", label="Conc (µM)")
    ax1.set_ylabel("Conc (µM)", fontsize=8, color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    ax2.plot(t, E, color="tab:red", label="Effect (%)")
    ax2.set_ylabel("Effect (%)", fontsize=8, color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")

    ax1.set_title(f'{row["drugbank_id"]}\n t½ = {t_half:.1f} h', fontsize=9)
    ax1.grid(True, alpha=0.4)

    # label x-axis only on last row (row index 4 → i >= 8)
    if i >= 8:
        ax1.set_xlabel("Time (h)", fontsize=8)

# Delete unused axes if <10
for j in range(len(top10), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("PK–PD with Varying Half-lives (Top 10 Compounds)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(10, 14), sharex=True)
axes = axes.ravel()  # now len(axes) == 10

# choose a colormap and sample N distinct colors
cmap = plt.cm.tab10  # or any other colormap
colors = cmap(np.linspace(0, 1, len(top10)))  # one color per compound

for i, ((_, row), t_half) in enumerate(zip(top10.iterrows(), t_half_values)):
    ax1 = axes[i]
    ax2 = ax1.twinx()

    k_el = np.log(2) / t_half
    C = C0 * np.exp(-k_el * t)
    E = E0 + (Emax * C**h) / (EC50**h + C**h)

    base_color = colors[i]

    # concentration curve: solid line
    ax1.plot(t, C, color=base_color, linestyle='-', label="Conc (µM)")
    ax1.set_ylabel("Conc (µM)", fontsize=8, color=base_color)
    ax1.tick_params(axis="y", labelcolor=base_color)

    # effect curve: same color, different style (e.g., dashed)
    ax2.plot(t, E, color=base_color, linestyle='--', label="Effect (%)")
    ax2.set_ylabel("Effect (%)", fontsize=8, color=base_color)
    ax2.tick_params(axis="y", labelcolor=base_color)

    ax1.set_title(f'{row["drugbank_id"]}\n t½ = {t_half:.1f} h', fontsize=9)
    ax1.grid(True, alpha=0.4)

    if i >= 8:
        ax1.set_xlabel("Time (h)", fontsize=8)

# Delete unused axes if <10
for j in range(len(top10), len(axes)):
    fig.delaxes(axes[j])

# ax1.plot(t, C, color="tab:blue", linewidth=3.0, label="Conc (µM)")
# ax2.plot(t, E, color="tab:red",  linewidth=3.0, label="Effect (%)")


fig.suptitle("PK–PD with Varying Half-lives (Top 10 Compounds)", fontsize=14)
plt.tight_layout()
plt.show()


## Drug toxicity

In [ ]:
from rdkit import Chem
import pandas as pd

# Example: your dataframe with SMILES
# df = top10  # make sure df has a 'smiles' column

# 1) Define a small library of structural alerts (toy example)
# SMARTS are from common "toxicophore" motifs discussed in reviews (aromatic nitro, anilide, etc.)
alerts = [
    {
        "name": "Aromatic nitro",
        "endpoint": "Mutagenicity / carcinogenicity",
        "smarts": "[NX3](=O)=O-[c]"   # –NO2 attached to aromatic carbon
    },
    {
        "name": "Aniline",
        "endpoint": "Methemoglobinemia / hepatotoxicity",
        "smarts": "[NX3H2]-[c]"       # –NH2 attached to aromatic ring
    },
    {
        "name": "Aryl halide",
        "endpoint": "Reactive metabolite risk",
        "smarts": "[F,Cl,Br,I]-[c]"   # halogen on aromatic ring
    },
    {
        "name": "Michael acceptor",
        "endpoint": "Covalent binding / sensitization",
        "smarts": "C=CC=O"            # simple α,β‑unsaturated carbonyl
    },
]

# Pre‑compile SMARTS patterns
for a in alerts:
    a["pattern"] = Chem.MolFromSmarts(a["smarts"])

# 2) Function to scan one molecule
def find_alerts(smiles):
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        return []

    hits = []
    for a in alerts:
        if m.HasSubstructMatch(a["pattern"]):
            hits.append(a["name"])
    return hits

# 3) Apply to your table
df = top10.copy()                # or whatever df holds your SMILES
df["Toxicity_alerts"] = df["smiles"].apply(find_alerts)

# 4) Expand into a more readable table
alert_rows = []
for idx, row in df.iterrows():
    if not row["Toxicity_alerts"]:
        alert_rows.append({
            "index": idx,
            "smiles": row["smiles"],
            "alert_name": "None detected",
        })
    else:
        for name in row["Toxicity_alerts"]:
            alert_rows.append({
                "index": idx,
                "smiles": row["smiles"],
                "alert_name": name,
            })

alerts_table = pd.DataFrame(alert_rows)

print(df[["drugbank_id", "Toxicity_alerts"]])
print(alerts_table)


## Plots and Visualizations

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Compute ROC curve points
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("ROC Curve for Activity Classifier")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


This ROC curve shows that our classifier ranks actives better than random, with an AUC of about 0.72 indicating moderate discrimination power.


In [ ]:
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# import matplotlib.pyplot as plt

# cm = confusion_matrix(y_test, y_pred)  # from clf.predict(X_test)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
# disp.plot(cmap=plt.cm.Blues)
# plt.title("Confusion Matrix: Activity Classifier")
# plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1) Sort test set by predicted probability (descending)
order = np.argsort(-y_prob)
y_sorted = y_test[order]

# 2) Cumulative fraction of actives vs fraction of dataset screened
n = len(y_sorted)
frac_screened = np.arange(1, n + 1) / n                    # x-axis
cum_actives = np.cumsum(y_sorted)                          # cumulative count of actives
total_actives = cum_actives[-1]
frac_actives_found = cum_actives / total_actives           # y-axis

plt.figure(figsize=(5, 5))
plt.plot(frac_screened * 100, frac_actives_found * 100, label="Model")
plt.plot([0, 100], [0, 100], "k--", label="Random")        # random baseline
plt.xlabel("Percent of compounds screened")
plt.ylabel("Percent of actives found")
plt.title("Enrichment / Accumulation Curve")
plt.xlim(0, 10)    # focus on early 0–10%; remove or change if you want full range
plt.ylim(0, 100)
plt.grid(True)
plt.legend()
plt.show()


This enrichment curve shows that by screening only the top 10% of compounds ranked by your model, you recover about 33–35% of all actives, whereas random screening would recover only ~10%

Labels (active/inactive): These come from the real pIC50 and the threshold (e.g., pIC50 ≥ 8.2 → active = 1, else inactive = 0). They are ground‑truth, not used for ranking.
​

Ranking: The model gives each compound a score (probability of being active or predicted pIC50), and you sort all compounds by this score and take the top 10 % of the sorted list, then ask “how many of these top‑ranked compounds are truly active according to the labels?”.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# df_cls already has columns: 'active', 'molecular_weight', 'logp', 'HBA', 'HBD', 'RotBonds', 'AromaticRings'

desc_cols = ["molecular_weight", "logp", "HBA", "HBD", "RotBonds", "AromaticRings"]

# 1) Histograms (actives vs inactives)
for col in desc_cols:
    plt.figure(figsize=(5,4))
    sns.histplot(data=df_cls, x=col, hue="active", bins=30, stat="density", common_norm=False)
    plt.title(f"Histogram of {col}: actives vs inactives")
    plt.xlabel(col)
    plt.ylabel("Density")
    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

# Use the same subset for embeddings and labels
df_vis = df_cls.dropna(subset=["smiles"]).reset_index(drop=True)
smiles_vis = df_vis["smiles"].tolist()
y = df_vis["active"].values          # length N_vis

# Recompute embeddings for this subset
emb_list = []
for i in range(0, len(smiles_vis), batch_size):
    batch = smiles_vis[i:i+batch_size]
    emb = embed_smiles(batch)
    emb_list.append(emb)
X_vis = np.vstack(emb_list)          # shape (N_vis, hidden_dim)

# t-SNE
tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=0)
X_2d = tsne.fit_transform(X_vis)

# Explicit class colors: 0 -> blue, 1 -> red
colors = np.where(y == 1, "red", "blue")

# Plot
plt.figure(figsize=(6,5))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=colors, alpha=0.7, s=10)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE visualization of ChemBERTa embeddings")

# Manual legend matching colors
from matplotlib.lines import Line2D
legend_elems = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=6, label='Inactive (0)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red',  markersize=6, label='Active (1)')
]
plt.legend(handles=legend_elems, loc="best")

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # needed for 3D plots
import numpy as np

# df_vis, smiles_vis, y, X_vis as before
df_vis = df_cls.dropna(subset=["smiles"]).reset_index(drop=True)
smiles_vis = df_vis["smiles"].tolist()
y = df_vis["active"].values

# embeddings
emb_list = []
for i in range(0, len(smiles_vis), batch_size):
    batch = smiles_vis[i:i+batch_size]
    emb = embed_smiles(batch)
    emb_list.append(emb)
X_vis = np.vstack(emb_list)

# 3D t-SNE
tsne3 = TSNE(n_components=3, perplexity=30, learning_rate=200, random_state=0)
X_3d = tsne3.fit_transform(X_vis)

# colors: 0 -> blue, 1 -> red
colors = np.where(y == 1, "red", "blue")

fig = plt.figure(figsize=(7,6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_3d[:,0], X_3d[:,1], X_3d[:,2],
           c=colors, s=10, alpha=0.7)

ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_zlabel("t-SNE 3")
ax.set_title("3D t-SNE of ChemBERTa embeddings")

from matplotlib.lines import Line2D
legend_elems = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=6, label='Inactive (0)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red',  markersize=6, label='Active (1)')
]
ax.legend(handles=legend_elems, loc="best")

plt.tight_layout()
plt.show()


In [ ]:
# Assuming your dataframe is named df
top10_smiles = df[["drugbank_id", "smiles"]]
top10_smiles.to_csv("top10_smiles.csv", index=False)


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import os

top = df[["drugbank_id","smiles"]].drop_duplicates()
os.makedirs("ligands_sdf", exist_ok=True)

for _, row in top.iterrows():
    dbid = row["drugbank_id"]
    smi = row["smiles"]
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print("Failed SMILES for", dbid)
        continue
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=0xf00d)
    AllChem.UFFOptimizeMolecule(mol)
    Chem.MolToMolFile(mol, f"ligands_sdf/{dbid}.sdf")


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import os

top = df[["drugbank_id","smiles"]].drop_duplicates()  # df from your notebook
os.makedirs("ligands_sdf", exist_ok=True)

for _, row in top.iterrows():
    dbid = row["drugbank_id"]
    smi = row["smiles"]
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print("Failed SMILES for", dbid)
        continue
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=0xf00d)
    AllChem.UFFOptimizeMolecule(mol)
    Chem.MolToMolFile(mol, f"ligands_sdf/{dbid}.sdf")


In [ ]:
import glob, os, pandas as pd

logs = glob.glob("docking_out_sdf/*.log")
print("Example logs:", [os.path.basename(x) for x in logs[:5]])


## Computing pIC50 values for other drugs Donepezil, Rivastigmine, Galantamine for caspase-4

In [ ]:
import pandas as pd
import numpy as np

# Your three reference drugs and Vina scores (kcal/mol)
ref_drugs = pd.DataFrame([
    {"Drug": "Donepezil",   "DrugBank_ID": "DB00843",
     "SMILES": "COc1ccc2c(c1)C(=O)C(CC1CCN(CCc3ccccc3)CC1)C2",
     "Score_kcal_mol": -8.69},
    {"Drug": "Rivastigmine","DrugBank_ID": "DB00989",
     "SMILES": "CCOC(=O)NCC(C)OC1=CC=CC=C1",
     "Score_kcal_mol": -6.70},
    {"Drug": "Galantamine", "DrugBank_ID": "DB00674",
     "SMILES": "COc1ccc2c(c1)[C@H]3CC[C@H](C[C@H]3N2C)O",
     "Score_kcal_mol": -6.63},
])

# ΔG (Vina score) -> Ki -> pIC50
R = 1.987e-3      # kcal/(mol·K)
T = 298.15        # K
RT = R * T        # ≈ 0.592 kcal/mol

def dg_to_pIC50(dg_kcal_mol, rt=RT):
    Ki = np.exp(dg_kcal_mol / rt)   # mol/L
    return -np.log10(Ki)

ref_drugs["pIC50_model"] = ref_drugs["Score_kcal_mol"].apply(dg_to_pIC50)

print(ref_drugs)


## Alternative method: Computing pIC50 values for other drugs Donepezil, Rivastigmine, Galantamine for caspase-4

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd
import numpy as np

molecule = new_client.molecule

drug_names = ["Donepezil", "Rivastigmine", "Galantamine"]

chembl_map = {}
for name in drug_names:
    hits = list(molecule.search(name))   # <-- no cross_references here
    print(name, "hits:", len(hits))
    if hits:
        chembl_id = hits[0]["molecule_chembl_id"]
        chembl_map[name] = chembl_id
        print("  using", chembl_id)


In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd
import numpy as np

# -----------------------------
# 1. Configuration
# -----------------------------
# ChEMBL IDs for your three drugs
chembl_map = {
    "Donepezil":    "CHEMBL502",
    "Rivastigmine": "CHEMBL636",
    "Galantamine":  "CHEMBL1555",
}

# Caspase-4 target ChEMBL ID
target_id = "CHEMBL1075276"

# -----------------------------
# 2. Helper: IC50 -> pIC50
# -----------------------------
def ic50_to_pIC50(value, units):
    """
    Convert IC50 value + units to pIC50.
    value: numeric IC50
    units: 'nM', 'uM', or 'M'
    """
    if value is None:
        return np.nan
    v = float(value)
    if units == "nM":
        M = v * 1e-9
    elif units == "uM":
        M = v * 1e-6
    elif units == "M":
        M = v
    else:
        return np.nan
    if M <= 0:
        return np.nan
    return -np.log10(M)

# -----------------------------
# 3. Query ChEMBL activities
# -----------------------------
activity = new_client.activity

rows = []
for drug_name, chembl_id in chembl_map.items():
    print(f"Querying IC50 for {drug_name} ({chembl_id}) on {target_id}...")
    
    acts = activity.filter(
        molecule_chembl_id=chembl_id,
        target_chembl_id=target_id,
        standard_type="IC50"
    )
    
    for a in acts:
        rows.append({
            "Drug": drug_name,
            "molecule_chembl_id": chembl_id,
            "standard_value": a["standard_value"],
            "standard_units": a["standard_units"],
            "pchembl_value": a.get("pchembl_value"),
            "assay_chembl_id": a["assay_chembl_id"],
        })

df_ic50 = pd.DataFrame(rows)
print("\nRaw IC50 records:")
print(df_ic50)

# -----------------------------
# 4. Compute experimental pIC50
# -----------------------------
if not df_ic50.empty:
    if "pchembl_value" in df_ic50.columns:
        df_ic50["pIC50_exp"] = df_ic50["pchembl_value"].combine_first(
            df_ic50.apply(
                lambda r: ic50_to_pIC50(r["standard_value"], r["standard_units"]),
                axis=1
            )
        )
    else:
        df_ic50["pIC50_exp"] = df_ic50.apply(
            lambda r: ic50_to_pIC50(r["standard_value"], r["standard_units"]),
            axis=1
        )

    print("\nWith experimental pIC50:")
    print(df_ic50)
else:
    print("\nNo IC50 data found in ChEMBL for these drugs on caspase-4.")
